In [1]:
import os
import sys

import logging



logging.getLogger("snowflake").setLevel(logging.WARNING)
logging.getLogger("snowflake.connector").setLevel(logging.WARNING)
logging.getLogger("snowflake.snowpark").setLevel(logging.WARNING)





In [2]:
%pwd
os.chdir("../")

# Add the absolute path to src/ so Python can find automatch
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.append(src_path)
    
%pwd

'c:\\Users\\fiscarelli\\Desktop\\Progetti\\Manpower IT\\Auto-Match\\Candidates-to-Jobs-Auto-Match-Cortex-AI'

In [3]:
from src.autoMatch.utils.snowflake_utils import get_snowpark_session
session = get_snowpark_session()

In [4]:
from src.autoMatch.config.configuration import ConfigurationManager
from src.autoMatch.components.data_transformation import DataTransformation

import json
from snowflake.snowpark.functions import col, lit, trim, call_function, concat

config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)

input_table = data_transformation_config.candidate.input_table
education_levels = data_transformation_config.education_levels
regions = data_transformation_config.regions
ner_features = data_transformation_config.candidate.ner_features




[2026-04-02 11:36:20,838: INFO: common: yaml file: C:\Users\fiscarelli\Desktop\Progetti\Manpower IT\Auto-Match\Candidates-to-Jobs-Auto-Match-Cortex-AI\config\config_test.yaml loaded successfully]
[2026-04-02 11:36:20,845: INFO: common: yaml file: C:\Users\fiscarelli\Desktop\Progetti\Manpower IT\Auto-Match\Candidates-to-Jobs-Auto-Match-Cortex-AI\config\params.yaml loaded successfully]
[2026-04-02 11:36:20,861: INFO: common: yaml file: C:\Users\fiscarelli\Desktop\Progetti\Manpower IT\Auto-Match\Candidates-to-Jobs-Auto-Match-Cortex-AI\config\schema.yaml loaded successfully]


In [ ]:
from datetime import date, datetime
today_string = date.today().strftime("%Y-%m-%d")


system_prompt = f"""
                Estrai dal seguente testo i campi: 
                age (stringa. calcolala basandoti su date_of_birth, considerando che la data di oggi e {today_string};
                    esempio: se date_of_birth = 1990-05-18 e oggi siamo in data 2025-09-15, age = 35 
                    Se non è possibile calcolarla, restituisci stringa vuota), 
                date_of_birth (stringa in formato YYYY-MM-DD, 
                    YYYY deve essere compreso tra 1900 e {today_string},
                    MM deve essere compreso tra 1 e 12,
                    DD deve essere compreso tra: 
                        1 e 30 quando MM uguale a 11, 04, 06, 09;
                        1 e 28 quando MM uguale a 02
                        1 e 31 per i restanti casi
                    ), 
                location (stringa. il domicilio. solo la citta, non includere province o altro. Esempio: Milano), 
                region (la regione corrispondente alla citta del domicilio. stringa. possibili valori (solo uno): {", ".join(regions)},
                country (la nazione corrispondente alla citta del domicilio. stringa, solo in Italiano. Esempio: Italia),
                zip_code (cap. 5 caratteri numerici. Esempio: 20100), 
                last_job (Esempio: Data Engineer), 
                second_last_job (Esempio: Ingegnere del Software), 
                third_last_job (Esempio: Intern), 
                skills (skills tecniche, stringa. Esempio: Python, SQL, Java, Programmazione ad oggetti),
                soft_skills (stringa),
                languages (stringa, solo in Italiano.  Solo la lingua, non aggiungere livelli o aggettivi. Esempio: Italiano, Inglese, Hindi),
                certifications (stringa. Esempio: CISSP, EIPASS, ECDL, PEKIT, ESOL, MOS),
                education (stringa, solo in Italiano, possibili valori (solo uno): {", ".join(education_levels)})',
                'Rispondi in formato JSON, senza testo extra, non usare virgolette all interno dei campi. attieniti a questo esempio: 
                {{"age": 30, "date_of_birth": "1993-05-12", "location": "Milano", "region": "Lombardia", "country": "Italia", "zip_code": "20100", 
                "last_job": "Data Engineer", "second_last_job": "Ingegnere del Software", "third_last_job": "Intern", 
                "skills": "Python, SQL, Java",
                "soft_skills": "Precisione, Puntualità, Problem solving, Team Building, Gestione del tempo, Assistenza del cliente",
                "languages": "Italiano, Inglese, Hindi",
                "certifications": "CISSP, EIPASS, ECDL, PEKIT, ESOL, MOS",
                "education": "Diploma scuola superiore}}
            """

system_prompt



'\n                Estrai dal seguente testo i campi: \n                age (stringa. calcolala basandoti su date_of_birth, considerando che la data di oggi e 2026-03-06;\n                    esempio: se date_of_birth = 1990-05-18 e oggi siamo in data 2025-09-15, age = 35 \n                    Se non è possibile calcolarla, restituisci stringa vuota), \n                date_of_birth (stringa in formato YYYY-MM-DD, \n                    YYYY deve essere compreso tra 1900 e 2026-03-06,\n                    MM deve essere compreso tra 1 e 12,\n                    DD deve essere compreso tra: \n                        1 e 30 quando MM uguale a 11, 04, 06, 09;\n                        1 e 28 quando MM uguale a 02\n                        1 e 31 per i restanti casi\n                    ), \n                location (stringa. il domicilio. solo la citta, non includere province o altro. Esempio: Milano), \n                region (la regione corrispondente alla citta del domicilio. stringa. pos

In [ ]:


regions_json = json.dumps(regions)  # lista Python → JSON
education_levels_json = json.dumps(education_levels)


task_schema = f"""
{{
  "schema": {{
    "type": "object",
    "properties": {{
      "age": {{
        "type": "string",
        "description": "Restituisci solo un numero. calcolala basandoti su date_of_birth, considerando che la data di oggi e {today_string}; esempio: se la data di nascita è 1990-05-18 e oggi siamo in data 2025-09-15, age = 35. Se non è possibile calcolarla, restituisci stringa vuota
      }},
      "date_of_birth": {{
        "type": "string",
        "description": "date_of_birth (stringa in formato YYYY-MM-DD"
      }},
      "city": {{
        "type": "string",
        "description": "città di domicilio. solo la citta, non includere province o altro"
      }},
      "region": {{
        "type": "string",
        "description": "la regione corrispondente alla citta del domicilio. Possibili valori (solo uno): {", ".join(regions)}"
      }},
      "country": {{
        "type": "string",
        "description": "la nazione corrispondente alla citta del domicilio. solo in Italiano. Esempio: Italia, Spagna, Iran etc.)"
      }},
      "zip_code": {{
        "type": "string",
        "description": "cap. 5 caratteri numerici"
      }},
      "last_job": {{
        "type": "string",
        "description": "L ultimo lavoro ricoperto. Ad esempio: Data Engineer"
      }},
      "second_last_job": {{
        "type": "string",
        "description": "Il penultimo lavoro ricoperto. Ad esempio: Software developer"
      }},
      "third_last_job": {{
        "type": "string",
        "description": "Il terzo ultimo lavoro ricoperto. Ad esempio: Project Manager"
      }},
      "skills": {{
        "type": "string",
        "description": "skills tecniche, separate da virgola. Ad esempio: Python, SQL, Java"
      }},
      "soft_skills": {{
        "type": "string",
        "description": "soft skills, separate da virgola. Ad esempio: Precisione, Puntualità, Problem solving"
      }},
      "languages": {{
        "type": "string",
        "description": "lingue conosciute dal candidato, separate da virgola. Ad esempio: Italiano, Inglese"
      }},
      "certifications": {{
        "type": "string",
        "description": "certificazioni, separate da virgola. Ad esempio: CISSP, EIPASS, ECDL, PEKIT, ESOL, MOS"
      }},
      "education": {{
        "type": "string",
        "description": "solo in Italiano, possibili valori (solo uno): {", ".join(education_levels)}"
      }}
    }}
  }}
}}
""" #% today_string


'''
task_schema = f"""
{{'location': 'Città di domicilio', 'age': 'Eta del candidato'}}
"""
'''

task_json = json.dumps(task_schema)

task_schema




'\n{\n  "schema": {\n    "type": "object",\n    "properties": {\n      "age": {\n        "type": "string",\n        "description": "Restituisci solo un numero. calcolala basandoti su date_of_birth, considerando che la data di oggi e 2026-03-06; esempio: se la data di nascita è 1990-05-18 e oggi siamo in data 2025-09-15, age = 35. Se non è possibile calcolarla, restituisci stringa vuota\n      },\n      "date_of_birth": {\n        "type": "string",\n        "description": "date_of_birth (stringa in formato YYYY-MM-DD"\n      },\n      "city": {\n        "type": "string",\n        "description": "città di domicilio. solo la citta, non includere province o altro"\n      },\n      "region": {\n        "type": "string",\n        "description": "la regione corrispondente alla citta del domicilio. Possibili valori (solo uno): Abruzzo, Basilicata, Calabria, Campania, Emilia-Romagna, Friuli-Venezia Giulia, Lazio, Liguria, Lombardia, Marche, Molise, Piemonte, Puglia, Sardegna, Sicilia, Toscana, 

In [ ]:
from snowflake.snowpark.types import StringType, BooleanType
from snowflake.snowpark.functions import udf


def validate_json(df, json_column):
    
    json_column_new = 'json_cleaned'
    def build_clean_parsing_udf():
        def clean(x: str) -> str:
            if x is None:
                return ''
            x = x.lower().lstrip()
            if x.startswith("```json"):
                x = x[8:].lstrip()
            x = x.replace('\n', ' ').replace('\t', ' ').replace('\\', '').strip()
            x = ' '.join(x.split())
            if x.endswith("```"):
                x = x[:-3].rstrip()
            if x.endswith("'") or x.endswith('"'):
                x = x[:-1].rstrip()
            return x

        return udf(clean, return_type=StringType(), input_types=[StringType()])
    clean_udf = build_clean_parsing_udf()

    df = df.with_column(json_column_new, clean_udf(df[json_column]))

    def build_is_valid_json_udf():
        import json
        def is_valid(text: str) -> bool:
            if not text:
                return False
            try:
                json.loads(text)
                return True
            except Exception:
                return False

        return udf(is_valid, return_type=BooleanType(), input_types=[StringType()])
    
    is_valid_json_udf = build_is_valid_json_udf()
    df = df.with_column("is_valid_json", is_valid_json_udf(df[json_column_new]))
    #df = df.filter(col("is_valid_json") == True)
    #df = df.drop("is_valid_json")

    #df = df.with_column(json_column_new, parse_json(col(json_column_new)))
    from snowflake.snowpark.functions import call_function
    df = df.with_column(
        json_column_new,
        call_function("TRY_PARSE_JSON", col(json_column_new))
    )
    
    #df = df.filter(df[json_column_new].is_not_null())

    return df

In [ ]:
table_name = "test_model"
df = (
        session.table(f"{input_table}")
        .filter(
            (
                (col("description").is_not_null()) 
                & (trim(col("description")) != "")
            )
            #& (len(col("description")) > 100)
            | (col("candidateid").isin([5960485, 5960483, 2011330, 5960411, 5937647, 5954942, 5960303, 3386034, 1742181, 4650889]))
            )
        .limit(100)
    )
#df.write.save_as_table(table_name, mode="overwrite")

print(df.count())

100


In [ ]:
input_table

'MPG_IT_AUTOMATCH_CANDIDATE_TEST'

In [ ]:
#cost for Million tokens

model_name = 'llama3.1-8b'
cost_in = 0.11
cost_out = 0.11

model_name = 'llama4-maverick'
cost_in = 0.12
cost_out = 0.49

model_name = 'openai-gpt-5'
cost_in = 0.69
cost_out = 5.5

model_name = 'openai-gpt-5-mini'
cost_in = 0.14
cost_out = 1.1


table_model_name = table_name+'_'+model_name.replace("-", "_").replace(".", "d")
"""
df_extracted = (
    session.table(table_name)
    .select(
        "*",
            call_function(
                "SNOWFLAKE.CORTEX.AI_EXTRACT",
                col("DESCRIPTION"),
                call_function("PARSE_JSON", lit(task_schema))
            ).alias("EXTRACTED_JSON")
            )
)
"""
df_extracted = (
    session.table(table_name)
    .select(
        "*",
            call_function(
                "AI_COMPLETE",
                lit(model_name),
                concat(
                        lit(system_prompt),
                        lit("\nTesto: "),
                        col("description")
                    )
            ).alias("EXTRACTED_JSON")
            )
)

df_extracted = validate_json(df_extracted, "EXTRACTED_JSON")




df_extracted = df_extracted.select(
    "*",
    *[col("json_cleaned")[field].cast("STRING").alias(field) for field in ner_features]

)

df_extracted.write.save_as_table(table_model_name, mode="overwrite")


In [ ]:

df_final = (
    session.table(table_model_name)
    .with_column(
        "TOKEN_COUNT_INPUT",
        call_function(
            "SNOWFLAKE.CORTEX.AI_COUNT_TOKENS",
            lit("AI_COMPLETE"),
            lit(model_name),
            concat(
                lit(system_prompt),
                lit("\nTesto: "),
                col("description")
            )
        )
    )
    .with_column(
        "TOKEN_COUNT_OUTPUT",
        call_function(
            "SNOWFLAKE.CORTEX.AI_COUNT_TOKENS",
            lit("AI_COMPLETE"),
            lit(model_name),
            concat(col("EXTRACTED_JSON"))
        )
    )

)


In [ ]:
df_final = (
    df_final
    .with_column("TOKEN_COUNT_TOTAL", col("TOKEN_COUNT_INPUT") + col("TOKEN_COUNT_OUTPUT"))
    .with_column("CREDITS_INPUT", lit(1/1000000)*col("TOKEN_COUNT_INPUT")*lit(cost_in))
    .with_column("CREDITS_OUTPUT", lit(1/1000000)*col("TOKEN_COUNT_OUTPUT")*lit(cost_out))
    .with_column("CREDITS_TOTAL", col("CREDITS_INPUT") + col("CREDITS_OUTPUT"))
)
df_final.write.save_as_table(table_model_name, mode="overwrite")
